# MotoGP Same Nation Podium Lockouts - Dataset Exploration

This notebook explores the podium lockouts dataset showing races where all three podium positions were occupied by riders from the same nation.

## 0. Notebook Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading and Structure

In [ ]:
# Load data from CSV
data_path = Path("../data/raw/same_nation_podium_lockouts.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 15 rows:")
df.head(15)

In [ ]:
print(f"Columns: {list(df.columns)}")
print(f"Data types:\n{df.dtypes}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nUnique values per column:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

## 2. National Dominance Analysis

In [ ]:
# Nations with most podium lockouts
print("=== NATIONS WITH MOST PODIUM LOCKOUTS ===")
nation_lockouts = df['Riders` Nation'].value_counts()
print("Total podium lockouts by nation:")
print(nation_lockouts)

# Pie chart of national dominance
plt.figure(figsize=(10, 8))
plt.pie(nation_lockouts.values, labels=nation_lockouts.index, 
        autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Podium Lockouts by Nation')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 3. Temporal Analysis

In [ ]:
# Lockouts by season
print("=== PODIUM LOCKOUTS BY SEASON ===")
season_lockouts = df['Season'].value_counts().sort_index()
print(f"Time period: {df['Season'].min()} to {df['Season'].max()}")
print(f"Total seasons with lockouts: {df['Season'].nunique()}")
print(f"Average lockouts per season: {len(df) / df['Season'].nunique():.2f}")

print("\nSeasons with most lockouts:")
print(season_lockouts.nlargest(10))

# Time series plot
plt.figure(figsize=(15, 8))
plt.plot(season_lockouts.index, season_lockouts.values, marker='o', linewidth=2)
plt.xlabel('Season')
plt.ylabel('Number of Podium Lockouts')
plt.title('Podium Lockouts Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Track Analysis

In [ ]:
# Tracks with most lockouts
print("=== TRACKS WITH MOST PODIUM LOCKOUTS ===")
track_lockouts = df['Track'].value_counts()
print("Top 15 tracks with most lockouts:")
print(track_lockouts.head(15))

# Bar chart of top tracks
plt.figure(figsize=(14, 8))
top_tracks = track_lockouts.head(10)
plt.bar(range(len(top_tracks)), top_tracks.values, color='lightgreen')
plt.xticks(range(len(top_tracks)), 
           [track[:20] + '...' if len(track) > 20 else track for track in top_tracks.index], 
           rotation=45, ha='right')
plt.xlabel('Track')
plt.ylabel('Number of Lockouts')
plt.title('Top 10 Tracks by Number of Podium Lockouts')
plt.tight_layout()
plt.show()

## 5. Class Analysis

In [ ]:
# Lockouts by class
print("=== PODIUM LOCKOUTS BY CLASS ===")
class_lockouts = df['Class'].value_counts()
print("Lockouts by racing class:")
print(class_lockouts)

# Nation-Class combinations
print("\n=== NATION-CLASS LOCKOUT COMBINATIONS ===")
nation_class = df.groupby(['Riders` Nation', 'Class']).size().unstack(fill_value=0)
print("Lockouts by nation and class:")
print(nation_class)

# Heatmap of nation-class combinations
plt.figure(figsize=(12, 8))
sns.heatmap(nation_class, annot=True, fmt='d', cmap='YlOrRd', 
            cbar_kws={'label': 'Number of Lockouts'})
plt.title('Podium Lockouts by Nation and Class')
plt.xlabel('Class')
plt.ylabel('Nation')
plt.tight_layout()
plt.show()

## 6. Detailed Nation Analysis

In [ ]:
# Detailed analysis per dominant nation
print("=== DETAILED ANALYSIS BY DOMINANT NATIONS ===")
for nation in nation_lockouts.head(5).index:
    nation_data = df[df['Riders` Nation'] == nation]
    print(f"\n{nation} Analysis:")
    print(f"  Total lockouts: {len(nation_data)}")
    print(f"  Season range: {nation_data['Season'].min()}-{nation_data['Season'].max()}")
    print(f"  Most successful tracks:")
    top_tracks_nation = nation_data['Track'].value_counts().head(5)
    for track, count in top_tracks_nation.items():
        print(f"    {track}: {count} lockouts")
    print(f"  Classes dominated:")
    classes_nation = nation_data['Class'].value_counts()
    for class_name, count in classes_nation.items():
        print(f"    {class_name}: {count} lockouts")

## 7. Timeline Analysis by Nation

In [ ]:
# Timeline of lockouts by top nations
plt.figure(figsize=(15, 10))
top_nations = nation_lockouts.head(5).index

for nation in top_nations:
    nation_timeline = df[df['Riders` Nation'] == nation]['Season'].value_counts().sort_index()
    plt.plot(nation_timeline.index, nation_timeline.values, 
             marker='o', linewidth=2, label=nation)

plt.xlabel('Season')
plt.ylabel('Number of Lockouts')
plt.title('Timeline of Podium Lockouts by Top 5 Nations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Recent vs Historical Analysis

In [ ]:
# Compare recent years vs historical
print("=== RECENT VS HISTORICAL ANALYSIS ===")
median_year = df['Season'].median()
recent_data = df[df['Season'] > median_year]
historical_data = df[df['Season'] <= median_year]

print(f"Dividing at year {median_year}")
print(f"Historical period: {historical_data['Season'].min()}-{int(median_year)} ({len(historical_data)} lockouts)")
print(f"Recent period: {int(median_year)+1}-{recent_data['Season'].max()} ({len(recent_data)} lockouts)")

print("\nHistorical period top nations:")
historical_nations = historical_data['Riders` Nation'].value_counts().head(5)
print(historical_nations)

print("\nRecent period top nations:")
recent_nations = recent_data['Riders` Nation'].value_counts().head(5)
print(recent_nations)

# Side by side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

ax1.pie(historical_nations.values, labels=historical_nations.index, autopct='%1.1f%%', startangle=90)
ax1.set_title(f'Historical Period ({historical_data["Season"].min()}-{int(median_year)})')

ax2.pie(recent_nations.values, labels=recent_nations.index, autopct='%1.1f%%', startangle=90)
ax2.set_title(f'Recent Period ({int(median_year)+1}-{recent_data["Season"].max()})')

plt.tight_layout()
plt.show()

## 9. Key Insights Summary

Based on the exploration of the podium lockouts dataset:

### Dataset Characteristics
- Records instances where all three podium positions went to riders from same nation
- Covers multiple seasons, tracks, and racing classes
- Provides insights into periods of national dominance in motorcycle racing

### National Dominance Patterns
- **Clear Leaders**: Certain nations dominate podium lockout statistics
- **Class Specialization**: Some nations excel in specific racing classes
- **Track Preferences**: Particular tracks favor certain nations
- **Temporal Variations**: National dominance shifts over time periods

### Track Characteristics
- Some circuits consistently produce same-nation podiums
- Track characteristics may favor riders from certain countries
- Home advantage appears to play a role in some cases
- Certain venues have become synonymous with national dominance

### Temporal Trends
- Frequency of lockouts varies significantly by season
- Some periods show concentrated national dominance
- Recent vs historical patterns reveal changing competitive landscape
- Certain eras were defined by specific national supremacy

### Class-Specific Patterns
- Different classes show different national dominance patterns
- Some nations excel across multiple classes
- Others specialize in specific categories
- Class evolution affects national representation

This analysis provides foundation for answering business questions about national dominance, podium lockout patterns, and identifying which countries have achieved the most complete podium sweeps in MotoGP history.